In [1]:
import numpy as np

### Topic 35: Gradient Descent & Learning Rate ($\alpha$) Dynamics

* **The Update Rule:** $W_{new} = W_{old} - \alpha \nabla_W L$
* **The Learning Rate ($\alpha$):** Scales the gradient magnitude. It is the most critical hyperparameter in ML.
* **Hessian & Convergence Condition:** Strict convergence requires $\alpha < \frac{2}{\lambda_{max}}$ (where $\lambda_{max}$ is the largest eigenvalue of the Hessian).
* **Ill-Conditioned Valleys:** High condition numbers ($\frac{\lambda_{max}}{\lambda_{min}} \gg 1$) cause standard Gradient Descent to zigzag inefficiently across ravine walls.
* **NumPy Reality:** $W$ and $\nabla_W L$ are tensors. The update step must be a single, vectorized tensor subtraction. No loops over individual weights.

In [2]:
Q = np.array([[1.0,0.0],
              [0.0,50.0]])

def compute_loss(W):
    return (W.T @ Q @ W).item()

def compute_grad(W):
    return 2 * (Q @ W)

W = np.array([[5.0],
              [5.0]])

alpha = 0.015

print("initial weight: ",W)
print("compute loss: ",compute_loss(W))

grad = compute_grad(W)
W = W - alpha * grad
print("weight after 1 step: ",W)
print("weight after 1 step: ",compute_loss(W))

initial weight:  [[5.]
 [5.]]
compute loss:  1275.0
weight after 1 step:  [[ 4.85]
 [-2.5 ]]
weight after 1 step:  336.0225


In [3]:
Q = np.array([[1.0,0.0],
              [0.0,100.0]])

def compute_loss(W):
    return W.T @ Q @ W

def compute_grad(W):
    return 2 * Q @ W

W = np.array([[8.0],
              [8.0]])

print("initial weight: ",W)
print("compute loss: ",compute_loss(W))

alpha = 0.009

for i in range(30):
    grad = compute_grad(W)
    W = W - alpha * grad
    print(f"weight after {i + 1} step: ",compute_loss(W))

initial weight:  [[8.]
 [8.]]
compute loss:  [[6464.]]
weight after 1 step:  [[4157.716736]]
weight after 2 step:  [[2680.95492973]]
weight after 3 step:  [[1735.11327509]]
weight after 4 step:  [[1129.08599369]]
weight after 5 step:  [[740.56447845]]
weight after 6 step:  [[491.27034439]]
weight after 7 step:  [[331.10457992]]
weight after 8 step:  [[228.00300258]]
weight after 9 step:  [[161.44374964]]
weight after 10 step:  [[118.29207102]]
weight after 11 step:  [[90.14099579]]
weight after 12 step:  [[71.60935776]]
weight after 13 step:  [[59.25253087]]
weight after 14 step:  [[50.86529906]]
weight after 15 step:  [[45.035692]]
weight after 16 step:  [[40.85943919]]
weight after 17 step:  [[37.75721979]]
weight after 18 step:  [[35.35770166]]
weight after 19 step:  [[33.4226857]]
weight after 20 step:  [[31.79919742]]
weight after 21 step:  [[30.38882491]]
weight after 22 step:  [[29.12809441]]
weight after 23 step:  [[27.9759101]]
weight after 24 step:  [[26.90551486]]
weight aft

loss function is $L(W) = W^T Q W$. The second derivative (the Hessian matrix $H$) is $2Q$:$$H = \begin{bmatrix} 2 & 0 \\ 0 & 200 \end{bmatrix}$$The update step is $W_{new} = W_{old} - \alpha (H W_{old}) = (I - \alpha H)W_{old}$.For this sequence to converge to zero, the multiplier $(1 - \alpha \lambda_i)$ for every eigenvalue $\lambda_i$ of the Hessian must have an absolute value less than 1.Looking at the steep axis ($w_2$), the eigenvalue is $\lambda_{max} = 200$.$$|1 - 200\alpha| < 1 \implies \alpha < \frac{2}{200} \implies \alpha < 0.01$$By setting $\alpha = 0.17$, your multiplier for the $w_2$ axis becomes $1 - 200(0.17) = -33$.In each step, the $w_2$ parameter isn't descending; it is flipping signs and multiplying by 33. By step 30, $w_2$ evaluates to $8.0 \times (-33)^{30}$, which causes a 64-bit float overflow.
* choose an $\alpha < 0.01$. If you test $\alpha = 0.009$, the loss will stabilize and decrease.

### Topic 36: Convexity & Optimization Traps

* **Convex Functions:** Hessian is positive definite everywhere. Guarantee: Local Minimum == Global Minimum (e.g., Linear/Logistic Regression).
* **Non-Convex Functions:** Complex landscapes with multiple valleys. Deep Neural Networks are highly non-convex.
* **The Traps:** Standard Gradient Descent is greedy. It will blindly walk into the nearest local minimum and stop because $\nabla L = 0$.
* **Saddle Points:** Points where $\nabla L = 0$, but it is a minimum along some axes and a maximum along others. In high-dimensional Deep Learning, saddle points stall training far more frequently than true local minima.

In [7]:
import numpy as np

def compute_loss_nc(w):
    # L(w) = w^4 - 4w^2 + w + 10
    return w**4 - 4 * w**2 + w + 10

def compute_grad_nc(w):
    # dL/dw = 4w^3 - 8w + 1
    return 4 * w**3 - 8 * w + 1

# Let's test two different starting points
w_init_1 = -3.0 # Starts on the left side
w_init_2 = 3.0  # Starts on the right side
alpha = 0.01

def run_gd(w_current, steps=20):
    for _ in range(steps):
        grad = compute_grad_nc(w_current)
        w_current = w_current - alpha * grad
    return w_current

final_w_1 = run_gd(w_init_1)
final_w_2 = run_gd(w_init_2)

print(f"Start W: -3.0 -> Converged to W: {final_w_1:.4f} (Global Min, Loss: {compute_loss_nc(final_w_1):.4f})")
print(f"Start W:  3.0 -> Converged to W: {final_w_2:.4f} (Local Min, Loss: {compute_loss_nc(final_w_2):.4f})")

Start W: -3.0 -> Converged to W: -1.4812 (Global Min, Loss: 4.5564)
Start W:  3.0 -> Converged to W: 1.3687 (Local Min, Loss: 7.3848)


In [14]:
def compute_loss_nc(w):
    return w[0,0]**2 - w[1,0]**2

def compute_grad_nc(w):
    # dL/dw = 4w^3 - 8w + 1
    return np.array([[2*w[0,0]],
                     [-2*w[1,0]]])

def run_gd(w_init, steps=30,alpha = 0.1):
    w_current = w_init.copy()
    for _ in range(steps):
        grad = compute_grad_nc(w_current)
        w_current = w_current - alpha * grad
    return w_current


w_init_1 = np.array([[0.0],
                     [0.0]])

w_noise  = np.array([[0.001],
                     [0.001]])

final_w_1 = run_gd(w_init_1)
final_w_noise = run_gd(w_noise)

print(f"Start W: -3.0 -> Converged to W: {final_w_1} (Global Min, Loss: {compute_loss_nc(final_w_1)})")

print("--- Scenario A: Zero Gradient Trap ---")
print(f"Start W:\n{w_init_1}")
print(f"Final W after 30 steps:\n{final_w_1}")
print(f"Final Loss: {compute_loss_nc(final_w_1):.6f}\n")

print("--- Scenario B: Symmetry Breaking Escape ---")
print(f"Start W:\n{w_noise}")
print(f"Final W after 30 steps:\n{final_w_noise}")
print(f"Final Loss: {compute_loss_nc(final_w_noise):.6f}")

Start W: -3.0 -> Converged to W: [[0.]
 [0.]] (Global Min, Loss: 0.0)
--- Scenario A: Zero Gradient Trap ---
Start W:
[[0.]
 [0.]]
Final W after 30 steps:
[[0.]
 [0.]]
Final Loss: 0.000000

--- Scenario B: Symmetry Breaking Escape ---
Start W:
[[0.001]
 [0.001]]
Final W after 30 steps:
[[1.23794004e-06]
 [2.37376314e-01]]
Final Loss: -0.056348
